In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/11010
1


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  2058.6444463816388
RUN  2 , total integrated cost =  1181.7927532693466
RUN  3 , total integrated cost =  825.1967627073229
RUN  4 , total integrated cost =  683.8990769301896
RUN  5 , total integrated cost =  623.0681353381821
RUN  6 , total integrated cost =  576.7274744428893
RUN  7 , total integrated cost =  555.1166508344978
RUN  8 , total integrated cost =  526.6926476093774
RUN  9 , total integrated cost =  512.7082160802295
RUN  10 , total integrated cost =  463.70165420440253
RUN  11 , total integrated cost =  453.16423485174937
RUN  12 , total integrated cost =  452.80969314108967
RUN  13 , total integrated cost =  440.1903987893803
RUN  14 , total integrated cost =  440.19039878937946
RUN  15 , total integrated cost =  440.19039878937923


ERROR:root:Problem in initial value trasfer


RUN  16 , total integrated cost =  440.1903987893791
RUN  17 , total integrated cost =  440.1903987893791
Control only changes marginally.
RUN  17 , total integrated cost =  440.1903987893791
Improved over  17  iterations in  20.486647188663483  seconds by  91.36422660618365  percent.
Problem in initial value trasfer:  Vmean_exc -65.23383853135607 -65.27384438522054
weight =  115.79738772627474
set cost params:  1.0 115.79738772627474 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4838.415124001652
Gradient descend method:  None
RUN  1 , total integrated cost =  4420.9571992045785
RUN  2 , total integrated cost =  4214.560982087705
RUN  3 , total integrated cost =  4080.725271696074
RUN  4 , total integrated cost =  4078.848166278235
RUN  5 , total integrated cost =  4078.8134683330727
RUN  6 , total integrated cost =  4078.813100762265
RUN  7 , total integrated cost =  4078.8130956525965
RUN  8 , total integrated cost =  4078.8130955793554
RUN  9 , total i

ERROR:root:Problem in initial value trasfer


RUN  14 , total integrated cost =  4078.8130955785946
Control only changes marginally.
RUN  14 , total integrated cost =  4078.8130955785946
Improved over  14  iterations in  0.6009671483188868  seconds by  15.699397611729154  percent.
Problem in initial value trasfer:  Vmean_exc -56.63945339219338 -56.63928622518727
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  10813.88670502415
RUN  2 , total integrated cost =  10762.231728933857
RUN  3 , total integrated cost =  10760.576231318753


...

RUN  1 , total integrated cost =  22820.421536661106
Control only changes marginally.
RUN  1 , total integrated cost =  22820.421536661106
Improved over  1  iterations in  0.12723703682422638  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.699660713182055 -56.69972571539597
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  32548.271002956615
RUN  2 , total integrated cost =  32539.22364608721
RUN  3 , total integrated cost =  32538.564429615228
RUN  4 , total integrated cost =  32537.296220234304
RUN  5 , total integrated cost =  32536.746577743015
RUN  6 , total integrated cost =  32536.38748053385
RUN  7 , total integrated cost =  32536.16237756681
RUN  8 , total integrated cost =  32535.97282134291
RUN  9 , total integrated cost =  32535.899332849403
RUN  10 , total integ

ERROR:root:Problem in initial value trasfer


RUN  110 , total integrated cost =  32534.02744433109
Control only changes marginally.
RUN  112 , total integrated cost =  32534.027444331085
Improved over  112  iterations in  4.050587344914675  seconds by  2.271020888324088  percent.
Problem in initial value trasfer:  Vmean_exc -56.70377935632819 -56.703770407894496
weight =  10.232379475255643
set cost params:  1.0 10.232379475255643 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32540.886337679105
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  32540.886337679105
Control only changes marginally.
RUN  1 , total integrated cost =  32540.886337679105
Improved over  1  iterations in  0.12782749719917774  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70377935632819 -56.703770407894496


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
no solutions found
closest index  -1
set cost params:  1.0 10.0 0.0
all options tried already
-------  5 0.4000000000000001 0.40000000000000013
no solutions found
closest index

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
no convergence
weight =  1404.0304781832285
set cost params:  1.0 1404.0304781832285 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5749.596021401725
Gradient descend method:  None
RUN  1 , total integrated cost =  5749.510538403574
RUN  2 , total integrated cost =  5749.51053840357


ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  5749.510538403566
RUN  4 , total integrated cost =  5749.510538403566
Control only changes marginally.
RUN  4 , total integrated cost =  5749.510538403566
Improved over  4  iterations in  0.325068574398756  seconds by  0.0014867652934356101  percent.
Problem in initial value trasfer:  Vmean_exc -56.626702245209074 -56.626716521294625
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  143.71191318598778
set cost params:  1.0 143.71191318598778 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4269.393935289574
Gradient descend method:  None
RUN  1 , total integrated cost =  4241.97931278007
RUN  2 , total integrated cost =  4240.886274019003
RUN  3 , total integrated cost =  4240.854545481651
RUN  4 , total integrated cost =  4240.854050406942
RUN  5 , total integrated cost =  4240.8540368674585
RUN  6 , total integrated cost =  4240.854036494399
RUN  7 , total integrated cost =  4240.854036486061
RUN  8

ERROR:root:Problem in initial value trasfer


RUN  12 , total integrated cost =  4240.854036485831
Control only changes marginally.
RUN  12 , total integrated cost =  4240.854036485831
Improved over  12  iterations in  0.5460678469389677  seconds by  0.6684765855837469  percent.
Problem in initial value trasfer:  Vmean_exc -56.63347244495723 -56.633375420689305
-------  10 0.4250000000000001 0.42500000000000016
no convergence
weight =  114.42863833864946
set cost params:  1.0 114.42863833864946 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8408.199104305775
Gradient descend method:  None
RUN  1 , total integrated cost =  8406.039259731337
RUN  2 , total integrated cost =  8406.039259731335


...

RUN  50 , total integrated cost =  22878.19405644731
Control only changes marginally.
RUN  52 , total integrated cost =  22878.194056447308
Improved over  52  iterations in  1.969626434147358  seconds by  0.0018598189875120852  percent.
Problem in initial value trasfer:  Vmean_exc -56.699792930739406 -56.69989590544141
-------  100 0.4500000000000001 0.7750000000000005
converged for  100
-------  105 0.5750000000000002 0.7750000000000005
converged for  105
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  115 0.4250000000000001 0.8250000000000005
no convergence
weight =  15.971104471666258
set cost params:  1.0 15.971104471666258 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5278.338869421614
Gradient descend method:  None
RUN  1 , total integrated cost =  5277.783042864028
RUN  2 , total integrated cost =  5277.783042864025
RUN  3 , total integrated cost =  5277.783042864025
Control only changes marginally.


ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  5277.783042864025
Improved over  3  iterations in  0.24137422069907188  seconds by  0.01053033106320811  percent.
Problem in initial value trasfer:  Vmean_exc -56.62256372329335 -56.62256759681972
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
-------  125 0.47500000000000014 0.8500000000000005
no convergence
weight =  6.704248726420686
set cost params:  1.0 6.704248726420686 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13733.795300159518
Gradient descend method:  None
RUN  1 , total integrated cost =  13733.794005671416
RUN  2 , total integrated cost =  13733.771346160442
RUN  3 , total integrated cost =  13733.768304700332
RUN  4 , total integrated cost =  13733.767784160877
RUN  5 , total integrated cost =  13733.767651021182
RUN  6 , total integrated cost =  13733.755227268211
RUN  7 , total integrated cost =  13733.748603630602
RUN  8 , total integrated cost =  13733.727257917417
RUN  9 , total

ERROR:root:Problem in initial value trasfer


RUN  16 , total integrated cost =  13733.677643904106
Control only changes marginally.
RUN  16 , total integrated cost =  13733.677643904106
Improved over  16  iterations in  0.6754997707903385  seconds by  0.0008566914887069288  percent.
Problem in initial value trasfer:  Vmean_exc -56.67324475950737 -56.67339901064588
-------  130 0.6000000000000003 0.8500000000000005
converged for  130
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
-------  145 0.5750000000000002 0.9000000000000006
converged for  145


In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [True, True], [True, True], [True, True], [False, False], [False, False], [True, True], [True, True], [False, False], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4282.9525191111425
Gradient descend method:  None
RUN  1 , total integrated cost =  37.95059630490351
RUN  2 , total integrated cost =  21.275441627551213
RUN  3 , total integrated cost =  14.724241609605832
RUN  4 , total integrated cost =  12.855518050558427
RUN  5 , total integrated cost =  10.362753766314057
RUN  6 , total integrated cost =  10.037829044596808
RUN  7 , total integrated cost =  9.726768189353214
RUN  8 , total integrated cost =  9.219026428740712
RUN  9 , total integrated cost =  8.712331279912984
RUN  10 , total integrated cost =  7.450443809479547
RUN  11 , total integrated cost =  7.395340909184553
RUN  12 , total integrated cost =  7.372404498489966
RUN  13 , total integrated cost =  7.335372357803648
RUN  14 , total integrated cost =  7.3304738704717405
RUN  15 , total integrated cost =  7.1508680565790055
RUN 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  33 , total integrated cost =  6.707622015341531
Improved over  33  iterations in  5.126690885052085  seconds by  99.84338789689096  percent.
Problem in initial value trasfer:  Vmean_exc -67.78196621842973 -67.78599574443207
weight =  100
set cost params:  1.0 100.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  22.163345082824552
Gradient descend method:  HS
RUN  1 , total integrated cost =  21.71913355249914
RUN  2 , total integrated cost =  21.578332395980066
RUN  3 , total integrated cost =  21.42062934164954
RUN  4 , total integrated cost =  21.383400250307417
RUN  5 , total integrated cost =  21.38019820001645
RUN  6 , total integrated cost =  21.380198200016427
RUN  7 , total integrated cost =  21.380198200016412


ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  21.380198200016412
Control only changes marginally.
RUN  8 , total integrated cost =  21.380198200016412
Improved over  8  iterations in  1.2469231933355331  seconds by  3.53352294015869  percent.
Problem in initial value trasfer:  Vmean_exc -67.63027775846473 -67.6366950397766
weight =  23840.17200651494
set cost params:  1.0 23840.17200651494 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5015.534496808259
Gradient descend method:  None
RUN  1 , total integrated cost =  4617.587544926863
RUN  2 , total integrated cost =  4617.265745675105
RUN  3 , total integrated cost =  4617.255163792203
RUN  4 , total integrated cost =  4617.253860219635
RUN  5 , total integrated cost =  4617.253625672344
RUN  6 , total integrated cost =  4617.253286008637
RUN  7 , total integrated cost =  4617.25321347664
RUN  8 , total integrated cost =  4617.253205458039
RUN  9 , total integrated cost =  4617.253204398208
RUN  10 , total integrated 

...

Control only changes marginally.
RUN  47 , total integrated cost =  57.5062019439535
Improved over  47  iterations in  7.6918263155967  seconds by  99.82204872200204  percent.
Problem in initial value trasfer:  Vmean_exc -64.15593893065704 -64.1727537113393
weight =  100
set cost params:  1.0 100.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1546.0933151745671
Gradient descend method:  HS
RUN  1 , total integrated cost =  1361.8225144289222
RUN  2 , total integrated cost =  1326.9542484471667
RUN  3 , total integrated cost =  1326.3231389685932
RUN  4 , total integrated cost =  1326.30701098833
RUN  5 , total integrated cost =  1325.377838437217
RUN  6 , total integrated cost =  1325.3713265719575
RUN  7 , total integrated cost =  1325.3704890560907
RUN  8 , total integrated cost =  1325.3702205131033
RUN  9 , total integrated cost =  1325.3702195375286
RUN  10 , total integrated cost =  1325.3702195374556
RUN  11 , total integrated cost =  1325.37021953

ERROR:root:Problem in initial value trasfer



RUN  13 , total integrated cost =  1325.3702195374499
Improved over  13  iterations in  2.009065704420209  seconds by  14.276182004719146  percent.
Problem in initial value trasfer:  Vmean_exc -58.99006015969172 -58.9782411772516
weight =  2510.754902603428
set cost params:  1.0 2510.754902603428 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  31961.13973175314
Gradient descend method:  None
RUN  1 , total integrated cost =  30116.551505803014
RUN  2 , total integrated cost =  30098.13285166132
RUN  3 , total integrated cost =  23488.8193925799
RUN  4 , total integrated cost =  21702.764840661184
RUN  5 , total integrated cost =  21486.960856207552
RUN  6 , total integrated cost =  21486.20476669422
RUN  7 , total integrated cost =  21486.20476363352
RUN  8 , total integrated cost =  21486.204763633516
RUN  9 , total integrated cost =  21486.204763633512
RUN  10 , total integrated cost =  21486.20476363351
RUN  11 , total integrated cost =  21486.2047636335

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  11 , total integrated cost =  21486.20476363351
Improved over  11  iterations in  1.6733380407094955  seconds by  32.77397194228611  percent.
Problem in initial value trasfer:  Vmean_exc -56.69303877884933 -56.69487451547576


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  26317.735598143136
set cost params:  1.0 26317.735598143136 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5085.962656701813
Gradient descend method:  None
RUN  1 , total integrated cost =  5085.5990024806215
RUN  2 , total integrated cost =  5085.584759838503
RUN  3 , total integrated cost =  5085.581336039596
RUN  4 , total integrated cost =  5085.5757046348535
RUN  5 , total integrated cost =  5085.57165648903
RUN  6 , total i

ERROR:root:Problem in initial value trasfer


RUN  19 , total integrated cost =  5085.566798390803
RUN  20 , total integrated cost =  5085.566798390803
Control only changes marginally.
RUN  20 , total integrated cost =  5085.566798390803
Improved over  20  iterations in  2.9390948191285133  seconds by  0.007783350719023474  percent.
Problem in initial value trasfer:  Vmean_exc -61.276221655781264 -61.316640685404835
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  7070.911110128155
set cost params:  1.0 7070.911110128155 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10312.315706139168
Gradient descend method:  None
RUN  1 , total integrated cost =  10048.389511719057
RUN  2 , total integrated cost =  10047.368606527525
RUN  3 , total integrated cost =  10047.367909387136
RUN  4 , total integrated cost =  10047.367908326465
RUN  5 , total integrated cost =  10047.367908325434
RUN  6 , total integrated cost =  10047.367908325428


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  10047.367908325421
RUN  8 , total integrated cost =  10047.367908325421
Control only changes marginally.
RUN  8 , total integrated cost =  10047.367908325421
Improved over  8  iterations in  1.1115283574908972  seconds by  2.569236681301547  percent.
Problem in initial value trasfer:  Vmean_exc -56.647804785505485 -56.648494752235656
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  10479.453128794261
set cost params:  1.0 10479.453128794261 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8227.714984045377
Gradient descend method:  None
RUN  1 , total integrated cost =  8227.648697729635
RUN  2 , total integrated cost =  8227.648530914432
RUN  3 , total integrated cost =  8227.648530914415
RUN  4 , total integrated cost =  8227.648530914405
RUN  5 , total integrated cost =  8227.648530914403


...

RUN  5 , total integrated cost =  14410.164189338806
State only changes marginally.
RUN  6 , total integrated cost =  14410.164189338806
Control only changes marginally.
RUN  6 , total integrated cost =  14410.164189338806
Improved over  6  iterations in  1.0774413105100393  seconds by  8.970637581739993e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.67648658702815 -56.67652202884036
-------  135 0.5250000000000001 0.8750000000000006
no convergence
weight =  141051.88621167076
set cost params:  1.0 141051.88621167076 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23309.93574829165
Gradient descend method:  None
RUN  1 , total integrated cost =  23309.9156271886
RUN  2 , total integrated cost =  23309.915627188573
RUN  3 , total integrated cost =  23309.91562718857
RUN  4 , total integrated cost =  23309.915627188566


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  23309.915627188566
Control only changes marginally.
RUN  5 , total integrated cost =  23309.915627188566
Improved over  5  iterations in  1.1573205851018429  seconds by  8.631985647866713e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.70034253990981 -56.700366414062465
-------  145 0.5750000000000002 0.9000000000000006
no convergence
weight =  121015.16652437014
set cost params:  1.0 121015.16652437014 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32974.17071983027
Gradient descend method:  None
RUN  1 , total integrated cost =  32974.14051427
RUN  2 , total integrated cost =  32974.14051426998
RUN  3 , total integrated cost =  32974.14051426997


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  32974.14051426997
Control only changes marginally.
RUN  4 , total integrated cost =  32974.14051426997
Improved over  4  iterations in  0.9810938686132431  seconds by  9.160369961591641e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.70368216673635 -56.70366928682795
--------------- 101


In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  121.5550889486953
Gradient descend method:  None
RUN  1 , total integrated cost =  0.35391802372827835
RUN  2 , total integrated cost =  0.3530034331587005
RUN  3 , total integrated cost =  0.3530027222301444
RUN  4 , total integrated cost =  0.353002722230143


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  0.3530027222301416
RUN  6 , total integrated cost =  0.3530027222301416
Control only changes marginally.
RUN  6 , total integrated cost =  0.3530027222301416
Improved over  6  iterations in  0.3885568082332611  seconds by  99.70959445196151  percent.
Problem in initial value trasfer:  Vmean_exc -67.84359095470359 -67.84707148873186
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12773.394245040696
Gradient descend method:  None
RUN  1 , total integrated cost =  9198.595853780671
RUN  2 , total integrated cost =  9084.435188705103
RUN  3 , total integrated cost =  9081.66765477274
RUN  4 , total integrated cost =  9081.230804692643
RUN  5 , total integrated cost =  9081.004443937132
RUN  6 , total integrated cost =  9080.850698380218
RUN  7 , total integrated cost =  9080.756891253215
RUN  8 , total integrated cost =  9080.694163388353
RUN  9 , total 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  39 , total integrated cost =  9080.519528695691
Improved over  39  iterations in  1.5372014474123716  seconds by  28.910676719923316  percent.
Problem in initial value trasfer:  Vmean_exc -56.64401036241986 -56.64456556163353
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  79.16424534389014
Gradient descend method:  None
RUN  1 , total integrated cost =  1.0954404744904442
RUN  2 , total integrated cost =  1.0954404744904422


...

RUN  100 , total integrated cost =  13141.8523842417
Control only changes marginally.
RUN  103 , total integrated cost =  13141.852384241696
Improved over  103  iterations in  2.237823350355029  seconds by  7.921210102389736  percent.
Problem in initial value trasfer:  Vmean_exc -56.67083936253185 -56.67107594346471
-------  135 0.5250000000000001 0.8750000000000006
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23087.79614494741
Gradient descend method:  None
RUN  1 , total integrated cost =  22058.0907656103
RUN  2 , total integrated cost =  22023.232977316653
RUN  3 , total integrated cost =  22019.898885411654
RUN  4 , total integrated cost =  22017.844641452175
RUN  5 , total integrated cost =  22016.201980571674
RUN  6 , total integrated cost =  22014.98888650574
RUN  7 , total integrated cost =  22014.27577740574
RUN  8 , total integrated cost =  22013.80244498213
RUN  9 , total integrated cost =  22013.29050717009
RUN  10 ,

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  99 , total integrated cost =  22001.135609770485
Improved over  99  iterations in  3.1044617649167776  seconds by  4.706644706817258  percent.
Problem in initial value trasfer:  Vmean_exc -56.698786496151484 -56.69890418923987
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32660.08619126245
Gradient descend method:  None
RUN  1 , total integrated cost =  31649.06276456259
RUN  2 , total integrated cost =  31623.313656586703
RUN  3 , total integrated cost =  31621.63926531273
RUN  4 , total integrated cost =  31620.497116080627
RUN  5 , total integrated cost =  31619.80331448811
RUN  6 , total integrated cost =  31618.72386439405
RUN  7 , total integrated cost =  31617.864817657734
RUN  8 , total integrated cost =  31617.179513577863
RUN  9 , total integrated cost =  31616.658414792713
RUN  10 , total integrated cost =  31616.16815624237
RUN  11

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  79 , total integrated cost =  31607.97169801789
Improved over  79  iterations in  2.5517992544919252  seconds by  3.2214075831986975  percent.
Problem in initial value trasfer:  Vmean_exc -56.70394077081876 -56.70392857182899


In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

ERROR:root:Problem in initial value trasfer


--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.3530027222301416
Gradient descend method:  None
RUN  1 , total integrated cost =  0.3530027222301416
Control only changes marginally.
RUN  1 , total integrated cost =  0.3530027222301416
Improved over  1  iterations in  0.1114125195890665  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.84359095470359 -67.84707148873186
-------  15 0.45000

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  9080.519528695691
Control only changes marginally.
RUN  1 , total integrated cost =  9080.519528695691
Improved over  1  iterations in  0.12477228790521622  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.64401036241986 -56.64456556163353
-------  25 0.4250000000000001 0.5000000000000002
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.0954404744904407
Gradient descend method:  None
RUN  1 , total integrated cost =  1.0954404744904407
Control only changes marginally.
RUN  1 , total integrated cost =  1.0954404744904407
Improved over  1  iterations in  0.11141377687454224  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -70.44978864433908 -70.47238247751108
-------  45 0.5000000000000002 0.5750000000000003
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  17583.79798045

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


...

RUN  1 , total integrated cost =  22032.3462319826
Control only changes marginally.
RUN  1 , total integrated cost =  22032.3462319826
Improved over  1  iterations in  0.12377642281353474  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.69879447399526 -56.69894197734752
-------  115 0.4250000000000001 0.8250000000000005
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.0026942712699485
Gradient descend method:  None
RUN  1 , total integrated cost =  1.0026942712699485
Control only changes marginally.
RUN  1 , total integrated cost =  1.0026942712699485
Improved over  1  iterations in  0.11335310898721218  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -73.78017325529072 -73.82388300285969
-------  125 0.47500000000000014 0.8500000000000005
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13141.8523842

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  13141.852384241696
Control only changes marginally.
RUN  1 , total integrated cost =  13141.852384241696
Improved over  1  iterations in  0.12175348028540611  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.67083936253185 -56.67107594346471
-------  135 0.5250000000000001 0.8750000000000006
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  22001.135609770485
Gradient descend method:  None
RUN  1 , total integrated cost =  22001.135609770485
Control only changes marginally.
RUN  1 , total integrated cost =  22001.135609770485
Improved over  1  iterations in  0.12014464847743511  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.698786496151484 -56.69890418923987
-------  145 0.5750000000000002 0.9000000000000006
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  31607.971

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  31607.97169801789
Control only changes marginally.
RUN  1 , total integrated cost =  31607.97169801789
Improved over  1  iterations in  0.12051297724246979  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70394077081876 -56.70392857182899
--------------- 12
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002 0.

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
